# Bài tập về nhà Buổi 1 - Đại số tuyến tính cho AI
**Họ và tên:** Phan Gia Huy
**Lựa chọn dữ liệu:** Text

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Text
cau = [
    "tôi thích học lập trình c++",
    "học máy và trí tuệ nhân tạo rất thú vị",
    "tôi muốn làm ai engineer trong tương lai",
    "cấu trúc dữ liệu và giải thuật c++",
    "mô hình ngôn ngữ lớn trong trí tuệ nhân tạo",
    "học lập trình python cho học máy",
    "thuật toán và cấu trúc dữ liệu",
    "ứng dụng của ai trong tương lai"
]

# Tạo vocab và hàm chuyển câu thành vector
vocab = sorted({w for s in cau for w in s.lower().split()})

def to_vector(s):
    v = np.zeros(len(vocab))
    for w in s.lower().split():
        if w in vocab:
            v[vocab.index(w)] += 1
    return v

# Ma trận X
X = np.array([to_vector(s) for s in cau])
print("Shape của ma trận X:", X.shape)

**Nhận xét:** Ma trận `X` có kích thước `(số_câu, số_từ_vựng)`. Ở đây là (8, 32). 
Mỗi hàng đại diện cho 1 câu trong tập dữ liệu. Mỗi cột đại diện cho 1 từ trong từ điển (vocab). Giá trị tại ô (i, j) là số lần từ thứ j xuất hiện trong câu thứ i.

In [ ]:
# BÀI 1: TÍNH TOÁN & ĐỘ TƯƠNG ĐỒNG

# Tính vector trung bình theo cột và trừ trung bình
mean_vec = np.mean(X, axis=0)
X_centered = X - mean_vec

print("Shape của X ban đầu:", X.shape)
print("Shape của mean_vec:", mean_vec.shape)
print("Shape của X_centered (sau khi broadcasting):", X_centered.shape)

# Hàm cosine similarity
def cosine_similarity(X_in, Y_in=None):
    if Y_in is None:
        Y_in = X_in
    # Chuẩn hóa
    X_n = X_in / (np.linalg.norm(X_in, axis=1, keepdims=True) + 1e-9)
    Y_n = Y_in / (np.linalg.norm(Y_in, axis=1, keepdims=True) + 1e-9)
    return X_n @ Y_n.T

# Hàm truy vấn
def search(query_idx, top_k=3):
    query_vec = X[query_idx:query_idx+1]
    sim_scores = cosine_similarity(query_vec, X)[0]
    
    # Lấy index giảm dần
    top_indices = np.argsort(sim_scores)[::-1]
    top_indices = [idx for idx in top_indices if idx != query_idx][:top_k]
    
    return top_indices, sim_scores[top_indices]

# Test search cho câu 0
top_idx, scores = search(0, top_k=3)
print(f"Câu truy vấn: '{cau[0]}'\n")
for i, idx in enumerate(top_idx):
    print(f"Top {i+1} giống nhất: '{cau[idx]}' (Score: {scores[i]:.4f})")

**Nhận xét:** Quy tắc broadcasting cho phép NumPy tự động mở rộng `mean_vec` (1D) thành 2D để khớp với `X` khi thực hiện phép trừ `X - mean_vec`.
Hàm tìm kiếm hoạt động tốt, các câu có chung nhóm từ vựng như "lập trình" hay "cấu trúc dữ liệu" có điểm Cosine cao hơn, phù hợp với trực giác.

In [ ]:
#  BÀI 2: BIẾN ĐỔI TUYẾN TÍNH & SVD (Text)

# Giảm chiều bằng SVD
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Đưa mỗi câu về 2 chiều
coords = U[:, :2] * S[:2]

# Trực quan hóa
plt.figure(figsize=(10, 6))
plt.scatter(coords[:, 0], coords[:, 1], c='blue', s=100)

for i in range(len(coords)):
    plt.annotate(f"  Câu {i}", (coords[i, 0], coords[i, 1]), fontsize=12)

plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.title("Biểu diễn 2D của các câu bằng SVD")
plt.xlabel("Trục 1")
plt.ylabel("Trục 2")
plt.grid(True)
plt.show()

**Nhận xét SVD:** Các câu cùng chủ đề (ví dụ: các câu thiên về "c++", "cấu trúc dữ liệu") sẽ nằm gần nhau tạo thành cụm, phân tách rõ ràng với cụm các câu về "ai", "học máy". Hai trục đại diện cho hai đặc trưng ẩn (latent features) mang lại lượng phương sai lớn nhất của dữ liệu, tương tự như trong PCA.

In [ ]:
# BONUS

def knn_1(query_text, X_train, train_labels):
    # Biến câu truy vấn thành vector
    q_vec = to_vector(query_text).reshape(1, -1)
    
    # Tính cosine similarity với toàn bộ tập train
    sim_scores = cosine_similarity(q_vec, X_train)[0]
    
    # Tìm index có điểm cao nhất
    best_idx = np.argmax(sim_scores)
    return train_labels[best_idx], sim_scores[best_idx]

# Giả lập nhãn cho 8 câu
labels = ["IT", "AI", "AI", "IT", "AI", "AI", "IT", "AI"]

# Test với 1 câu mới
cau_moi = "tôi học cấu trúc dữ liệu"
predicted_label, score = knn_1(cau_moi, X, labels)

print(f"Câu test: '{cau_moi}'")
print(f"Nhãn dự đoán (1-NN): {predicted_label} (Độ tự tin: {score:.4f})")